# 📊 Credit Default Prediction

## 🎯 Problema de Negócio
O objetivo é prever a probabilidade de inadimplência de clientes em cobranças mensais.

A inadimplência é definida como pagamentos realizados com atraso igual ou superior a 5 dias,
caracterizando um evento de risco relevante para o negócio.

---

## 🧠 Objetivo Técnico
Desenvolver um modelo de machine learning capaz de estimar a probabilidade de inadimplência
para cada cobrança futura, permitindo priorização de ações.

---

## ⚙️ Estratégia
- Construção da variável target a partir do histórico de pagamentos
- Criação de features comportamentais (lags, rolling e histórico do cliente)
- Uso de validação temporal para simular cenário real de produção
- Modelagem com algoritmo baseado em árvores (LightGBM)
- Avaliação com métricas adequadas para problemas desbalanceados (AUC, KS, AP)

---

## 💼 Aplicação
O modelo permite:
- Priorizar clientes com maior risco
- Otimizar ações de cobrança
- Reduzir custos operacionais
- Apoiar decisões de crédito

## 1. Setup do Ambiente


**Objetivo:** Centralizar todas as bibliotecas utilizadas no projeto, garantindo organização,
           reprodutibilidade e facilidade de manutenção do código.

**Decisão:** As bibliotecas foram escolhidas com base no padrão de mercado para problemas de modelagem preditiva com dados tabulares. Utiliza-se pandas e numpy para manipulação de dados, matplotlib e seaborn para visualização, e LightGBM como algoritmo principal por sua eficiência e desempenho em problemas de classificação.

**Métricas e validação:** Foram importadas métricas como AUC, KS e Precision-Recall para avaliar o modelo sob diferentes perspectivas, especialmente considerando possível desbalanceamento da target. Também foram incluídas ferramentas de calibração para garantir qualidade das probabilidades previstas.


In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    log_loss
)

from scipy.stats import ks_2samp

from sklearn.calibration import CalibratedClassifierCV, calibration_curve

## 2. Carregamento e Padronização das Bases de Dados


**Objetivo:** Carregar as bases de dados fornecidas e garantir consistência estrutural 
           para viabilizar integrações e transformações ao longo do pipeline.

**Decisão:** As bases foram carregadas separadamente respeitando sua granularidade:

                • cadastral → nível cliente (informações estáticas)
            
                • info → nível mensal (informações dinâmicas)
            
                • pagamentos → histórico com target
            
                • teste → base para inferência

Essa separação permite construir uma base analítica completa via joins controlados.


In [ ]:

PATH = '../data/'

cadastral = pd.read_csv(PATH + 'base_cadastral.csv', sep=';')
info = pd.read_csv(PATH + 'base_info.csv', sep=';')
pag = pd.read_csv(PATH + 'base_pagamentos_desenvolvimento.csv', sep=';')
teste = pd.read_csv(PATH + 'base_pagamentos_teste.csv', sep=';')

print("Arquivos carregados com sucesso!")



# Foi realizada a limpeza dos nomes das colunas para evitar inconsistências 
# (em especial espaços em branco), que podem gerar erros silenciosos em merges.


## Limpeza na base
cadastral.columns = cadastral.columns.str.strip()
info.columns = info.columns.str.strip()
pag.columns = pag.columns.str.strip()
teste.columns = teste.columns.str.strip()


# Uma verificação inicial das colunas garante que a estrutura está conforme esperado, evitando problemas nas etapas seguintes do pipeline.

print("\nColunas da base de pagamentos: ")
print(pag.columns)

## 3. Definição da Target de Inadimplência (Regra de Negócio)


**Objetivo:** Construir a variável target de inadimplência a partir da regra de negócio, transformando o problema em uma tarefa de classificação binária.

**Definição:** Considera-se inadimplente todo pagamento realizado com 5 dias ou mais de atraso em relação à data de vencimento.

**Decisão:** A definição da target foi baseada diretamente na regra do negócio, garantindo alinhamento entre o modelo preditivo e o problema real a ser resolvido.

**Tratamentos aplicados:**

    1. Conversão de datas: As colunas de data foram convertidas para datetime para permitir cálculos temporais precisos e evitar inconsistências.

    2. Cálculo de atraso: Foi criada a variável 'dias_atraso', representando o número de dias entre pagamento e vencimento, capturando o comportamento de pagamento do cliente.

    3. Tratamento de ausência de pagamento: Valores nulos em DATA_PAGAMENTO foram interpretados como não pagamento e tratados como atraso elevado (999 dias), garantindo correta classificação como inadimplente.

    4. Criação da target: 
                    A variável target foi definida como:
                            • 1 → atraso >= 5 dias
                            • 0 → caso contrário

    5. Variável auxiliar: Foi criada a variável 'atraso_grave' (>= 30 dias) para suportar análises exploratórias e possíveis estratégias futuras de segmentação de risco.

**Validação:** Foi realizada uma inspeção inicial da variável target para verificar sua distribuição, permitindo avaliar o nível de desbalanceamento do problema.


In [ ]:

# Conversão de datas
pag['DATA_PAGAMENTO'] = pd.to_datetime(pag['DATA_PAGAMENTO'], errors='coerce')
pag['DATA_VENCIMENTO'] = pd.to_datetime(pag['DATA_VENCIMENTO'], errors='coerce')

# Cálculo de atraso
pag['dias_atraso'] = (pag['DATA_PAGAMENTO'] - pag['DATA_VENCIMENTO']).dt.days

# Tratamento de não pagamento
# Se não pagou = vira atraso alto
pag['dias_atraso'] = pag['dias_atraso'].fillna(999)

# Criação da Target
pag['target'] = (pag['dias_atraso'] >= 5).astype(int)
pag['atraso_grave'] = (pag['dias_atraso'] >= 30).astype(int)

# Validação
print("\nAmostra dos dados:")
pag[['DATA_VENCIMENTO', 'DATA_PAGAMENTO', 'dias_atraso', 'target']].head(10)


print("\nDistribuição da target:")
print(pag['target'].value_counts(normalize=True))

## 4. Construção da Base Final


**Objetivo:** Construir a base analítica final consolidando todas as fontes de dados necessárias para modelagem.

**Estratégia:** A base de pagamentos foi utilizada como tabela principal (fato), pois contém a variável target e representa o evento de interesse. A partir dela, foram incorporadas informações complementares:

            • info → dados dinâmicos por cliente e período (join por ID_CLIENTE + SAFRA_REF)
            • cadastral → dados estáticos do cliente (join por ID_CLIENTE)

**Decisão:** Foi utilizado left join para preservar todos os registros da base de pagamentos, garantindo que nenhuma observação com target seja perdida.

**Ordenação:** A base foi ordenada por cliente e tempo, o que é essencial para etapas posteriores de engenharia de features baseadas em histórico (lags, rolling, etc.).



In [ ]:


df = pag.merge(info, on=['ID_CLIENTE', 'SAFRA_REF'], how='left')
df = df.merge(cadastral, on='ID_CLIENTE', how='left')

#  ordenação correta
df = df.sort_values(['ID_CLIENTE', 'SAFRA_REF', 'DATA_VENCIMENTO'])

df.head()

## 4.1 Validações e Sanity Checks


**Objetivo:** Validar a integridade da base após os merges, garantindo que a estrutura esteja consistente antes da etapa de engenharia de features.

**Decisão:** Foram realizadas verificações essenciais para evitar erros silenciosos que podem comprometer a modelagem, especialmente em problemas com múltiplas fontes de dados.

**Validações realizadas:**

    • Estrutura da base: Verificação de tipo e dimensão para garantir que os dados foram carregados e combinados corretamente.

    • Consistência de colunas: Confirmação da presença das variáveis esperadas após os merges.

    • Duplicidade: Checagem de duplicidade na chave (ID_CLIENTE + SAFRA_REF), assegurando granularidade correta (uma linha por cliente por período).

    • Valores ausentes: Análise do percentual de missing para identificar possíveis necessidades de tratamento na etapa de feature engineering.

    • Consistência temporal: Inspeção de registros de um cliente ao longo do tempo para garantir ordenação e coerência do histórico.

**Importância:** Essas validações são fundamentais para garantir a qualidade dos dados e evitar propagação de erros nas etapas seguintes, especialmente em features baseadas em histórico.


In [ ]:

print(type(df))
print(df.shape)

print("\nColunas:")
print(df.columns)

print("\nDuplicidades (ID_CLIENTE + SAFRA_REF):")
df[['ID_CLIENTE', 'SAFRA_REF']].duplicated().sum()

print("\nTop colunas com missing:")
df.isnull().mean().sort_values(ascending=False).head(10)

print("\nExemplo de registros:")
df[['ID_CLIENTE','SAFRA_REF']].head(10)


print("\nExemplo de histórico de um cliente:")
display(
    df[df['ID_CLIENTE'] == df['ID_CLIENTE'].iloc[0]][
        ['SAFRA_REF','DATA_VENCIMENTO']
    ].head(10)
)

## 5. Features Comportamentais (Histórico)


**Objetivo:** Criar features comportamentais baseadas no histórico do cliente, com foco em capturar padrões de inadimplência ao longo do tempo.

**Ideia:** A premissa central é que o comportamento passado do cliente é um forte preditor do comportamento futuro, especialmente em problemas de crédito.

**Decisão:** Todas as variáveis foram construídas utilizando apenas informações passadas, através do uso de shift(1), garantindo que não haja vazamento de informação (data leakage) e simulando corretamente um cenário de produção.

**Tipos de features criadas:**

    • Lags de inadimplência: Capturam o comportamento recente do cliente (ex: se inadimpliu nos últimos períodos).

    • Métricas de curto prazo (rolling): Médias móveis de inadimplência e atraso (3 períodos), capturando tendência recente.

    • Métricas de longo prazo: Janelas maiores (6 períodos) e médias acumuladas (expanding), capturando padrão estrutural de comportamento.

    • Perfil histórico: Taxas médias, atraso médio e volume de cobranças refletem o perfil de risco do cliente ao longo do tempo.

    • Valor financeiro: O valor médio pago pelo cliente foi incluído para capturar possível relação entre ticket médio e inadimplência.

    • Cliente novo: A flag identifica clientes sem histórico, diferenciando ausência de informação de comportamento efetivamente adimplente.

**Importância:** Essa abordagem permite combinar sinais de curto e longo prazo, aumentando o poder preditivo do modelo e mantendo consistência temporal.


In [ ]:


# Garantir ordenação correta antes de qualquer lag/rolling
df = df.sort_values(['ID_CLIENTE', 'SAFRA_REF', 'DATA_VENCIMENTO']).reset_index(drop=True)

## Lags de inadimplência 

# shift(1) dentro do groupby → só enxerga o passado do cliente
df['target_lag_1'] = df.groupby('ID_CLIENTE')['target'].shift(1)
df['target_lag_2'] = df.groupby('ID_CLIENTE')['target'].shift(2)
df['target_lag_3'] = df.groupby('ID_CLIENTE')['target'].shift(3)

# Rolling dentro do groupby (FIX: bug silencioso)
 
df['inad_ult_3'] = (
    df.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
 
df['atraso_ult_3'] = (
    df.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
 
# Média de atraso em janela de 6 meses (captura tendência de longo prazo)
df['atraso_ult_6'] = (
    df.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)

# Taxa histórica global por cliente 
# expanding() = média acumulada de todas as safras anteriores
df['taxa_hist_inad'] = (
    df.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).expanding().mean())
)
 
df['atraso_hist_medio'] = (
    df.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).expanding().mean())
)
 
# Total de cobranças anteriores do cliente
df['n_cobrancas_hist'] = (
    df.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).expanding().count())
)

 
df['valor_medio_cliente'] = (
    df.groupby('ID_CLIENTE')['VALOR_A_PAGAR']
    .transform(lambda x: x.shift(1).expanding().mean()) #usa só o passado
)
 
 
# Flag de cliente novo
# Criada ANTES do fillna para preservar a informação de ausência
# (cliente novo ≠ cliente com histórico limpo)
df['flag_cliente_novo'] = df['target_lag_1'].isna().astype(int)
 


## 5.1 Features de Contexto (Tempo e Cobrança)


**Objetivo:** Criar features de contexto relacionadas ao tempo de relacionamento e às características da cobrança atual.

Ideia: Além do histórico comportamental, o contexto em que a cobrança ocorre pode influenciar diretamente a probabilidade de inadimplência.

**Decisão:**

    • Tempo de relacionamento: A variável 'tempo_cliente' captura o tempo desde o cadastro até a data da cobrança. Clientes mais antigos tendem a ter comportamento mais estável, enquanto clientes recentes podem apresentar maior incerteza de risco.

    • Prazo da cobrança: A variável 'dias_prazo_cobranca' representa o tempo entre emissão e vencimento podendo influenciar a capacidade do cliente de se organizar para pagamento.

    • Sazonalidade: As variáveis 'mes_safra' e 'trimestre_safra' capturam padrões sazonais, permitindo ao modelo identificar períodos com maior propensão à inadimplência

**Importância:** Essas features adicionam uma visão contextual ao modelo, complementando o histórico do cliente com informações do momento da cobrança.



In [ ]:


# Tempo de relacionamento
df['DATA_CADASTRO'] = pd.to_datetime(df['DATA_CADASTRO'], errors='coerce')
df['tempo_cliente']  = (df['DATA_VENCIMENTO'] - df['DATA_CADASTRO']).dt.days
 
 
#  Features da cobrança atual
df['DATA_EMISSAO_DOCUMENTO'] = pd.to_datetime(df['DATA_EMISSAO_DOCUMENTO'], errors='coerce')
df['dias_prazo_cobranca']    = (df['DATA_VENCIMENTO'] - df['DATA_EMISSAO_DOCUMENTO']).dt.days
df['mes_safra']              = pd.to_datetime(df['SAFRA_REF']).dt.month
df['trimestre_safra']        = pd.to_datetime(df['SAFRA_REF']).dt.quarter

## 5.2 Tratamento de Missing Values


**Objetivo:** Tratar valores ausentes de forma consistente com o significado de cada variável, evitando distorções no modelo.

**Estratégia:**

    • Variáveis históricas → imputadas com 0 (interpretado como ausência de histórico do cliente)

    • Variáveis numéricas → imputadas com mediana (método robusto a outliers e distribuição assimétrica)

    • Variáveis categóricas → imputadas com 'DESCONHECIDO' (preserva informação de ausência sem introduzir viés artificial)

**Importância:** O tratamento adequado de valores ausentes evita perda de informação e garante maior estabilidade e performance do modelo.

In [ ]:

#  Imputação (FIX: fillna(0) indiscriminado) ─ Lags e rolling: NaN = sem histórico → imputar 0 faz sentido
COLS_LAG = [
    'target_lag_1', 'target_lag_2', 'target_lag_3',
    'inad_ult_3', 'atraso_ult_3', 'atraso_ult_6',
    'taxa_hist_inad', 'atraso_hist_medio', 'n_cobrancas_hist',
    'valor_medio_cliente',
]
df[COLS_LAG] = df[COLS_LAG].fillna(0)
 
# Demais numéricas: imputar pela mediana (não por 0)
for col in df.select_dtypes(include='number').columns:
    if col not in COLS_LAG and df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
 
# Categóricas: imputar com 'DESCONHECIDO'
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].fillna('DESCONHECIDO')


## 5.3 Validação Final


**Objetivo:** Validar a correta construção das features e garantir consistência antes da modelagem.

**Validações realizadas:**

    • Estrutura da base: Confirmação do tamanho final após criação das variáveis.

    • Inspeção manual: Visualização de amostras das principais features para verificar coerência dos valores,
                   especialmente variáveis baseadas em histórico (lags, rolling e acumuladas).

    • Consistência temporal: Verificação indireta de que as variáveis utilizam apenas informações passadas, garantindo ausência de data leakage.

**Importância:** Essa etapa reduz o risco de erros silenciosos na engenharia de features, que poderiam comprometer a performance e a confiabilidade do modelo.
              


In [ ]:

print("Features criadas sem data leakage - OK")
print(f"   Shape: {df.shape}")

print(df[['target_lag_1', 'inad_ult_3', 'atraso_ult_3',
          'taxa_hist_inad', 'valor_medio_cliente',
          'flag_cliente_novo']].head(8).to_string())
 

## 6. Split Temporal


**Objetivo:** Separar os dados em treino e validação respeitando a ordem temporal, garantindo consistência com um cenário real de previsão.

**Decisão:** Foi adotado um split temporal baseado na data de vencimento, onde o modelo é treinado apenas com dados passados e avaliado em períodos futuros.
         Essa abordagem evita vazamento de informação (data leakage), que ocorreria caso observações futuras influenciassem o treinamento do modelo.

**Justificativa:** Em problemas de crédito, o comportamento do cliente evolui ao longo do tempo. Utilizar um split aleatório poderia gerar métricas artificialmente infladas, pois o modelo teria acesso indireto a padrões do futuro.

Dessa forma, o split temporal simula o ambiente de produção, onde previsões são feitas com base exclusivamente em informações históricas.

**Validação:** Foi realizada uma verificação das proporções de inadimplência entre treino e validação, garantindo que não há distorções significativas entre os períodos.



In [ ]:

CUTOFF = '2018-10-01'

# Split temporal
train = df[df['DATA_VENCIMENTO'] < CUTOFF].copy()
val   = df[df['DATA_VENCIMENTO'] >= CUTOFF].copy()

# Sanity check 
print(f"\nTreino:    {len(train):,} linhas até {train['DATA_VENCIMENTO'].max().date()}")
print(f"Validação: {len(val):,} linhas a partir {val['DATA_VENCIMENTO'].min().date()}")
print(f"Taxa inadimplência treino: {train['target'].mean()*100:.1f}%")
print(f"Taxa inadimplência val:    {val['target'].mean()*100:.1f}%")


## 6.1 Definição das Features


**Objetivo:** Definir o conjunto final de variáveis (features) utilizadas no treinamento do modelo.

**Estratégia:** Foram removidas colunas que poderiam introduzir vazamento de informação, ruído ou identificação direta do cliente.

**Decisão:**

    • Identificadores: Colunas como ID_CLIENTE foram removidas, pois não possuem poder preditivo generalizável e podem induzir o modelo a memorizar padrões específicos.

    • Variáveis temporais brutas: Datas foram excluídas, pois não foram diretamente transformadas em sinais numéricos úteis. Em vez disso, foram utilizadas features derivadas (ex: mês, trimestre).

    • Variáveis com vazamento: Colunas como DATA_PAGAMENTO e dias_atraso contêm informação diretamente relacionada ao evento que queremos prever, o que causaria data leakage.

    • Variáveis derivadas da target: Variáveis como atraso_grave e a própria target foram removidas para evitar qualquer contaminação do modelo.

**Importância:** Essa etapa garante que o modelo utilize apenas informações disponíveis no momento da previsão, mantendo consistência com o cenário real de produção.



In [ ]:

COLS_EXCLUIR = [
    'ID_CLIENTE',
    'SAFRA_REF',
    'DATA_PAGAMENTO',
    'DATA_VENCIMENTO',
    'DATA_EMISSAO_DOCUMENTO',
    'DATA_CADASTRO',
    'dias_atraso',
    'atraso_grave',
    'target'
]

# garante que não há vazamento estrutural
FEATURES = [
    c for c in df.columns
    if c not in COLS_EXCLUIR
]

print(f"\nFeatures de modelagem ({len(FEATURES)}):")
print("\n".join(FEATURES))

## 6.2 Validação (Anti-leakage + Consistência)


**Objetivo:** Validar a consistência das features e reforçar a ausência de vazamento de informação antes do treinamento do modelo.

**Estratégia:** Foram realizadas verificações estruturais e semânticas nas variáveis utilizadas na modelagem.

**Validações realizadas:**

    • Checagem de vazamento: Busca por colunas contendo 'target' no nome, evitando inclusão acidental de variáveis diretamente relacionadas ao evento previsto.

    • Consistência entre treino e validação: Verificação das dimensões das bases para garantir que ambas utilizam o mesmo conjunto de features.

**Importância:** Essa etapa funciona como uma validação final do pipeline de modelagem, reduzindo o risco de erros silenciosos que poderiam comprometer a avaliação do modelo.

**Observação:** As features de histórico (ex: target_lag) são válidas, pois foram construídas utilizando apenas informações passadas (via shift),
não configurando vazamento de informação.


In [ ]:

## Checar se houve vazamento
leak_cols = [c for c in FEATURES if 'target' in c.lower()]
print("\nPossíveis vazamentos:", leak_cols)

## Validação de consistencia treino/val
print("\nDistribuição de features:")
print(train[FEATURES].shape)
print(val[FEATURES].shape)

## 7. Preparação do Modelo


**Objetivo:** Preparar os dados e configurar o modelo para o treinamento.

**Etapas e decisões:**

    • Separação de variáveis: Foram criadas as matrizes de treino e validação (X e y), garantindo uso consistente das features definidas anteriormente.

    • Tratamento de variáveis categóricas: As colunas categóricas foram convertidas para o tipo 'category', permitindo que o modelo (LightGBM) trate essas variáveis de forma otimizada, sem necessidade de encoding manual (ex: one-hot).

    • Tratamento de desbalanceamento: Foi utilizado o parâmetro scale_pos_weight, calculado a partir da proporção entre classes, para compensar o desbalanceamento da variável target, comum em problemas de inadimplência.

    • Escolha do modelo: O LightGBM foi escolhido por sua eficiência computacional e capacidade de capturar relações não lineares, além de lidar bem com grandes volumes de dados e variáveis heterogêneas.

    • Hiperparâmetros: Os parâmetros foram definidos buscando um equilíbrio entre performance e generalização, evitando overfitting (ex: subsample, colsample_bytree) e controlando a complexidade do modelo (ex: num_leaves, min_child_samples).

**Importância:** Essa preparação garante que o modelo seja treinado de forma robusta, considerando características reais do problema, como desbalanceamento e presença de variáveis categóricas.


In [ ]:

# Garantia de Definição (evita NameError)
TARGET = 'target'

# Matrizes de Treino e Validação
X_train = train[FEATURES].copy()
y_train = train[TARGET]

X_val   = val[FEATURES].copy()
y_val   = val[TARGET]


# Encoding Categóricas
cat_cols = X_train.select_dtypes(include=['object', 'string']).columns

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col]   = X_val[col].astype('category')


# Baleanceamento  
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw   = round(n_neg / n_pos, 2)

print(f"\nscale_pos_weight: {spw}")


# Modelo 
model = lgb.LGBMClassifier(
    n_estimators      = 1000,
    learning_rate     = 0.03,
    num_leaves        = 31,
    min_child_samples = 30,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = spw,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)



## 7.1 Treinando o Modelo


**Objetivo:** Treinar o modelo de classificação para previsão de inadimplência.

**Estratégia:** O modelo foi treinado utilizando LightGBM, com validação em conjunto separado temporalmente.

**Decisões:**

    • Uso de conjunto de validação: O modelo é avaliado durante o treinamento em dados fora do treino, permitindo monitorar capacidade de generalização.

    • Early stopping: Foi aplicado early stopping para interromper o treinamento quando não há ganho adicional na métrica de validação, evitando overfitting.

    • Monitoramento do treino: A evolução do modelo é acompanhada via logs periódicos, permitindo inspecionar estabilidade e convergência.

**Importância:** Essa abordagem garante que o modelo não apenas aprenda o padrão do treino, mas também mantenha desempenho consistente em dados futuros.



In [ ]:

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)

print(f"\nModelo treinado até a iteração: {model.best_iteration_}")


## 8. Avaliação Inicial do Modelo


**Objetivo:** Avaliar o desempenho inicial do modelo utilizando métricas adequadas para problemas de classificação desbalanceada.

**Métricas utilizadas:**

    • AUC-ROC: Mede a capacidade do modelo em distinguir entre clientes adimplentes e inadimplentes, independente do threshold de decisão.

    • KS (Kolmogorov-Smirnov): Avalia a separação entre as distribuições das classes (bons vs maus pagadores), sendo amplamente utilizada em modelos de crédito.

    • Average Precision (AP): Mede a qualidade das previsões considerando o desbalanceamento, focando na capacidade do modelo em identificar corretamente a classe positiva.

**Importância:** O uso combinado dessas métricas permite avaliar o modelo sob diferentes perspectivas, garantindo uma análise mais robusta do desempenho.

**Observação:** Neste estágio, o foco está na capacidade de ordenação (ranking) do modelo, e não na definição de um threshold específico de decisão.



In [ ]:

# Predição
prob_val = model.predict_proba(X_val)[:, 1]

# Métricas
auc = roc_auc_score(y_val, prob_val)
ap  = average_precision_score(y_val, prob_val)
ks, _ = ks_2samp(prob_val[y_val==1], prob_val[y_val==0])

print(f"""
========================
 Perfomace do Modelo
========================
AUC-ROC: {auc:.4f}
KS:      {ks:.4f}
AP:      {ap:.4f}
========================
""")


## 8.1 Avaliação Gráfica — Curvas ROC e Precision-Recall

**Objetivo:** Avaliar o desempenho do modelo de forma visual, analisando sua capacidade de separação e qualidade de ranking.

**Análises realizadas:**

    • Curva ROC: Avalia o trade-off entre taxa de verdadeiros positivos (TPR) e falsos positivos (FPR). Foi destacado o ponto de máximo ganho (Youden), útil como referência inicial de threshold.

    • Curva Precision-Recall: É mais adequada para cenários desbalanceados, permite avaliar o desempenho do modelo na identificação da classe positiva (inadimplência).

    • Distribuição dos scores: Comparação das distribuições de probabilidade entre adimplentes e inadimplentes, permitindo visualizar a separação entre as classes (base do KS).

**Importância:** Essas análises complementam as métricas numéricas, permitindo entender não apenas o desempenho global, mas também o comportamento do modelo ao longo dos diferentes níveis de score.

**Observação:** O foco permanece na capacidade de ordenação do modelo, sendo a escolha de threshold tratada posteriormente de forma alinhada ao contexto de negócio.



In [ ]:

# Curva ROC 
fpr, tpr, thresholds = roc_curve(y_val, prob_val)
idx_youden = (tpr - fpr).argmax()

# Curvas E Gráficos 

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Avaliação do modelo — validação temporal', fontsize=13)

# ROC
ax = axes[0]
ax.plot(fpr, tpr, label=f'AUC={auc:.3f}')
ax.scatter(fpr[idx_youden], tpr[idx_youden], color='red')
ax.plot([0, 1], [0, 1], '--')
ax.set_title('ROC')
ax.legend()

# Precision-Recall
prec, rec, _ = precision_recall_curve(y_val, prob_val)
ax = axes[1]
ax.plot(rec, prec, label=f'AP={ap:.3f}')
ax.axhline(y_val.mean(), linestyle='--')
ax.set_title('Precision-Recall')
ax.legend()

# Distribuição
ax = axes[2]
ax.hist(prob_val[y_val == 0], bins=40, alpha=0.6, label='Adimplente')
ax.hist(prob_val[y_val == 1], bins=40, alpha=0.6, label='Inadimplente')
ax.set_title(f'KS = {ks:.3f}')
ax.legend()

plt.tight_layout()
plt.show()


## 8.2 Avaliação com Matriz de Confusão


**Objetivo:** Avaliar o desempenho do modelo considerando um threshold específico, traduzindo probabilidades em decisões práticas (classificação).

**Estratégia:**

    • Definição de threshold: Foi utilizado o ponto de Youden como referência inicial para conversão das probabilidades em classes (adimplente vs inadimplente).

    • Matriz de confusão:
                    Permite analisar os acertos e erros do modelo em termos absolutos:
                        • Verdadeiros positivos (inadimplentes corretamente identificados)
                        • Falsos positivos (clientes bons classificados como risco)
                        • Falsos negativos (inadimplentes não identificados)

**Importância:** Essa análise conecta o modelo ao impacto de negócio, permitindo avaliar os diferentes tipos de erro e seus possíveis custos.

**Observação:** O threshold utilizado é uma referência estatística inicial; em produção, deve ser ajustado considerando trade-offs de negócio (ex: custo de cobrança, risco de inadimplência, experiência do cliente).



In [ ]:

# Matriz de Confusão 
threshold_otimo = thresholds[idx_youden]
y_pred = (prob_val >= threshold_otimo).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_val, y_pred),
    display_labels=['Adimplente', 'Inadimplente']
).plot(ax=ax, cmap='Blues', colorbar=False)

ax.set_title(f'Threshold = {threshold_otimo:.2f}')
plt.show()

## 8.3 Gains Chart (Priorização de Risco)


**Objetivo:** Avaliar a capacidade do modelo em priorizar clientes de maior risco, ordenando-os de acordo com a probabilidade prevista de inadimplência.

**Estratégia:** Os clientes foram ordenados do maior para o menor risco, permitindo analisar quanto da inadimplência total é capturada ao longo da população.

**Interpretação:**

             • Eixo X (% população): Representa a proporção de clientes analisados, começando pelos de maior risco.

             • Eixo Y (% inadimplentes capturados): Indica a fração acumulada de inadimplentes identificados.

             • Linha diagonal: Representa um modelo aleatório (baseline), onde não há capacidade de priorização.

**Importância:** O Gains Chart permite avaliar a eficiência operacional do modelo, mostrando se é possível capturar grande parte do risco analisando apenas uma parcela da base (ex: top 20% de maior risco).

**Observação:** Esse tipo de análise é especialmente relevante em cenários com restrição de recursos, onde nem todos os clientes podem ser abordados.



In [ ]:

# GAINS CHART 
df_gains = (
    pd.DataFrame({'prob': prob_val, 'target': y_val.values})
    .sort_values('prob', ascending=False)
)

df_gains['gains'] = df_gains['target'].cumsum() / df_gains['target'].sum()
df_gains['pop']   = (df_gains.index + 1) / len(df_gains)

plt.figure(figsize=(7,4))
plt.plot(df_gains['pop']*100, df_gains['gains']*100)
plt.plot([0,100],[0,100],'--')
plt.title('Gains Chart')
plt.xlabel('% população')
plt.ylabel('% inadimplentes capturados')
plt.show()

## 8.4 Importância das Variáveis no Modelo


**Objetivo:** Analisar a importância das variáveis no modelo, identificando quais features mais contribuíram para a previsão de inadimplência.

**Estratégia:** Foi utilizada a importância baseada em ganho do modelo (feature_importances_), que indica o quanto cada variável contribui para a redução de erro nas árvores.

**Interpretação:**

            • Features comportamentais: Variáveis como histórico de inadimplência e atraso tendem a aparecer entre as mais importantes, pois capturam diretamente o padrão de pagamento do cliente.

            • Features de contexto: Variáveis relacionadas ao tempo de relacionamento e características da cobrança complementam a análise, ajudando o modelo a entender o cenário da dívida.

**Importância:** Essa análise permite validar se o modelo está aprendendo padrões coerentes com o problema de negócio, além de aumentar a interpretabilidade.

**Observação:** A importância indica relevância relativa no modelo, mas não implica causalidade, devendo ser interpretada com cautela.



In [ ]:

importancia = pd.Series(model.feature_importances_, index=FEATURES)\
    .sort_values(ascending=False)

plt.figure(figsize=(8,5))
importancia.head(15).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Feature Importance')
plt.show()

print("\nTop 10 features:")
print(importancia.head(10))

## 9. Análise de Calibração das Probabilidades


**Objetivo:** Ajustar as probabilidades previstas pelo modelo para que reflitam melhor a frequência real de inadimplência.

**Estratégia:**

        • Probabilidades originais: Inicialmente, são obtidas as probabilidades previstas pelo modelo treinado, que possuem bom poder de ranking, mas podem não estar calibradas.

        • Calibração (Platt Scaling): Foi aplicada calibração utilizando o método sigmoid, que ajusta as probabilidades por meio de um modelo logístico auxiliar.

        • Validação interna (cv=3): A calibração é realizada com validação cruzada, evitando overfitting no ajuste das probabilidades.

**Importância:** Esse processo melhora a confiabilidade das probabilidades previstas, especialmente em cenários onde o score será utilizado para tomada de decisão, como definição de cutoff ou estratégias de cobrança.

**Observação:** A calibração atua sobre as probabilidades, não alterando significativamente o ranking do modelo, mas sim sua interpretação probabilística.


In [ ]:

#  Probalidades do Modelo Original
prob_val_raw = model.predict_proba(X_val)[:, 1]

# Calibração (Sigmoid / Platt Scalling)
calibrador = CalibratedClassifierCV(
    estimator=model,
    method='sigmoid',
    cv=3
)

calibrador.fit(X_train, y_train)

prob_val_cal = calibrador.predict_proba(X_val)[:, 1]


## 9.1 Avaliação da Calibração


**Objetivo:** Avaliar a qualidade da calibração das probabilidades e decidir se o modelo calibrado deve ser utilizado.

**Estratégia:**

    • Log Loss: Utilizado como métrica principal de calibração, pois penaliza probabilidades mal ajustadas. A comparação antes e depois indica se houve ganho real.

     • Curva de calibração (reliability curve): Permite avaliar visualmente o alinhamento entre probabilidades previstas e frequência observada. Quanto mais próxima da diagonal, melhor calibrado.

    • Distribuição das probabilidades: Análise complementar para verificar mudanças na dispersão dos scores após calibração.

**Decisão:** A calibração é adotada apenas se houver melhora no Log Loss, garantindo que o ajuste agrega valor sem degradar o modelo.

**Importância:** Essa etapa garante que as probabilidades sejam confiáveis para uso em decisões de negócio, como definição de threshold, priorização de clientes e gestão de risco.



In [ ]:


# LOG LOSS (Métrica Principal de Calibração)

ll_before = log_loss(y_val, prob_val_raw)
ll_after  = log_loss(y_val, prob_val_cal)

print("\n LOG LOSS (quanto menor melhor)")
print(f"Antes da calibração : {ll_before:.4f}")
print(f"Depois da calibração: {ll_after:.4f}")


# Curva de Calibração

prob_true_b, prob_pred_b = calibration_curve(y_val, prob_val_raw, n_bins=10)
prob_true_a, prob_pred_a = calibration_curve(y_val, prob_val_cal, n_bins=10)

plt.figure(figsize=(6,6))
plt.plot(prob_pred_b, prob_true_b, marker='o', label='Modelo original')
plt.plot(prob_pred_a, prob_true_a, marker='o', label='Modelo calibrado')
plt.plot([0,1], [0,1], '--', color='gray', label='Calibração perfeita')

plt.title("Curva de calibração (reliability diagram)")
plt.xlabel("Probabilidade prevista")
plt.ylabel("Frequência real")
plt.legend()
plt.grid()
plt.show()


# Distribuição das Probalidades

plt.figure(figsize=(6,4))
plt.hist(prob_val_raw, bins=50, alpha=0.5, label='Original')
plt.hist(prob_val_cal, bins=50, alpha=0.5, label='Calibrado')

plt.title("Distribuição das probabilidades")
plt.xlabel("Probabilidade")
plt.ylabel("Frequência")
plt.legend()
plt.show()


# Decisão Final

print("\n DECISÃO SOBRE CALIBRAÇÃO")

if ll_after < ll_before:
    print("Calibração melhorou o modelo → recomendada para produção")
else:
    print("Calibração NÃO melhorou o modelo")
    print("Decisão: manter probabilidades originais do modelo")

## 10. Validações e Sanity Checks do Modelo


**Objetivo:** Garantir que as previsões do modelo estão consistentes e confiáveis, evitando erros técnicos antes de sua utilização.

**Estratégia:**

        • Validação do range: As probabilidades devem estar no intervalo [0,1], garantindo coerência matemática.

        • Verificação de valores nulos: A presença de NaNs pode indicar falhas no pipeline ou inconsistências nos dados.

        • Consistência de tamanho: As predições devem ter correspondência exata com a base de validação.

        • Análise da distribuição do score: Avalia se o modelo apresenta variabilidade suficiente, evitando comportamentos degenerados (ex: prever valores constantes).

**Importância:** Esses checks funcionam como validações finais de integridade, prevenindo erros silenciosos e aumentando a confiabilidade do modelo para uso em produção.

**Observação:** Esse tipo de validação é comum em pipelines produtivos, sendo essencial para garantir robustez e estabilidade do sistema.



In [ ]:

# CHECK 1 — Range das Probabilidades
assert prob_val.min() >= 0 and prob_val.max() <= 1, \
    "Probabilidades fora do intervalo [0, 1]"

print("Check 1: probabilidades dentro do intervalo [0,1]")


# CHECK 2 — Não Existe NaN
assert np.isnan(prob_val).sum() == 0, \
    "Existem NaNs nas predições"

print("Check 2: sem NaNs nas probabilidades")


# CHECK 3 — Tamanho Consistente
assert len(prob_val) == len(X_val), \
    " Tamanho das predições não bate com validação"

print("Check 3: tamanho consistente")


# CHECK 4 — Estabilidade Básica do Score 
# (evita modelo degenerado ou travado)

std_score = np.std(prob_val)
mean_score = np.mean(prob_val)

assert std_score > 0.01, \
    " Modelo parece não variar (score colapsado)"

assert 0.01 < mean_score < 0.99, \
    "Média das probabilidades suspeita (muito extrema)"

print("Check 4: distribuição do score ok")

## 11. Gerando a Submissão


**Objetivo:** Preparar a base de teste para geração das previsões finais, replicando o mesmo pipeline aplicado na base de treino.

**Estratégia:**

        • Carregamento e limpeza: A base de teste é carregada e tratada para garantir consistência, incluindo remoção de duplicidades por cliente e safra.

        • Enriquecimento de dados: São realizados merges com as bases auxiliares (info e cadastral), mantendo a mesma estrutura utilizada na modelagem.

        • Padronização de datas: As colunas de data são convertidas para datetime, garantindo compatibilidade com as features temporais criadas anteriormente.

**Importância:** Essa etapa assegura que o modelo receba dados no mesmo formato utilizado no treinamento, evitando inconsistências e erros de predição.

**Observação:** Nenhuma informação futura ou variável de target é utilizada nesta etapa, preservando a integridade do processo e evitando data leakage.


In [ ]:

teste = pd.read_csv('../data/base_pagamentos_teste.csv', sep=';')
teste = teste.drop_duplicates(['ID_CLIENTE', 'SAFRA_REF'])

teste = teste.merge(info, on=['ID_CLIENTE', 'SAFRA_REF'], how='left')
teste = teste.merge(cadastral, on='ID_CLIENTE', how='left')

teste['DATA_VENCIMENTO'] = pd.to_datetime(teste['DATA_VENCIMENTO'], errors='coerce')
teste['DATA_EMISSAO_DOCUMENTO'] = pd.to_datetime(teste['DATA_EMISSAO_DOCUMENTO'], errors='coerce')
teste['DATA_CADASTRO'] = pd.to_datetime(teste['DATA_CADASTRO'], errors='coerce')

teste_feat = teste.copy()

print("Base de teste preparada")


## 11.1 Construção de Histórico (Sem Vazamento)


**Objetivo:** Construir variáveis históricas por cliente utilizando apenas informações disponíveis antes do período de previsão, evitando qualquer vazamento de dados.

**Estratégia:**

        • Restrição temporal: O histórico é construído utilizando apenas dados anteriores ao cutoff, garantindo que nenhuma informação futura seja utilizada.

        • Uso de lags e rolling: As variáveis são criadas com uso de shift(1), assegurando que apenas eventos passados influenciem as features atuais.

        • Agregação por cliente: As informações são consolidadas ao nível do cliente, capturando comportamento histórico sem misturar registros de diferentes períodos.

        • Separação de etapas: Lags e agregações são tratados separadamente para evitar inconsistências e garantir correta representação temporal.

**Importância:** Essa abordagem garante que as features utilizadas na base de teste refletem um cenário real de produção, onde apenas o histórico passado está disponível.

**Conclusão:** A construção do histórico foi realizada de forma robusta, preservando a integridade temporal e eliminando riscos de data leakage.


In [ ]:

hist_base = df[df['DATA_VENCIMENTO'] < CUTOFF].copy()
hist_base = hist_base.sort_values(['ID_CLIENTE', 'DATA_VENCIMENTO'])

# LAGS
for lag in [1,2,3]:
    hist_base[f'target_lag_{lag}'] = (
        hist_base.groupby('ID_CLIENTE')['target'].shift(lag)
    )

# Rolling Features
hist_base['inad_ult_3'] = (
    hist_base.groupby('ID_CLIENTE')['target']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

hist_base['atraso_ult_3'] = (
    hist_base.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

hist_base['atraso_ult_6'] = (
    hist_base.groupby('ID_CLIENTE')['dias_atraso']
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)

# Agregar sem usar Colunas Problemáticas
hist_cliente = (
    hist_base
    .groupby('ID_CLIENTE')
    .agg(
        taxa_hist_inad      = ('target', 'mean'),
        atraso_hist_medio   = ('dias_atraso', 'mean'),
        n_cobrancas_hist    = ('target', 'count'),
        valor_medio_cliente = ('VALOR_A_PAGAR', 'mean'),

        inad_ult_3          = ('inad_ult_3', 'last'),
        atraso_ult_3        = ('atraso_ult_3', 'last'),
        atraso_ult_6        = ('atraso_ult_6', 'last'),
    )
    .reset_index()
)

# LAGS Fora do AGG (Pega o Último Valor Separado)
lags = hist_base.groupby('ID_CLIENTE')[
    ['target_lag_1','target_lag_2','target_lag_3']
].last().reset_index()

hist_cliente = hist_cliente.merge(lags, on='ID_CLIENTE', how='left')


print("""
Histórico construído sem data leakage

Validações aplicadas:
- Uso de dados apenas anteriores ao cutoff temporal
- Aplicação de shift(1) para garantir uso exclusivo do passado
- Rolling e expanding respeitando ordem temporal
- Agregações por cliente (sem mistura de entidades)
""")

## 11.2 Aplicação das Features no Teste


**Objetivo:** Aplicar na base de teste as mesmas transformações realizadas no treino, garantindo consistência entre as etapas de modelagem e previsão.

**Estratégia:**

        • Enriquecimento com histórico: As features históricas por cliente são incorporadas via merge, utilizando apenas informações previamente construídas sem vazamento.

        • Identificação de cliente novo: A flag de cliente novo é criada para capturar ausência de histórico, permitindo ao modelo tratar esses casos de forma diferenciada.

        • Criação de features derivadas: São replicadas as variáveis de contexto temporal (tempo de relacionamento, prazo da cobrança e sazonalidade), mantendo alinhamento com o treino.

**Importância:** Essa etapa assegura que a base de teste possui exatamente o mesmo espaço de features utilizado pelo modelo, evitando inconsistências e erros de predição.

**Observação:** Clientes sem histórico recebem valores nulos nas variáveis agregadas, sendo tratados posteriormente na etapa de imputação.



In [ ]:

# Merge no Teste 
teste_feat = teste_feat.merge(hist_cliente, on='ID_CLIENTE', how='left')

teste_feat['flag_cliente_novo'] = teste_feat['taxa_hist_inad'].isna().astype(int)


# Features Derivadas

teste_feat['tempo_cliente'] = (
    teste_feat['DATA_VENCIMENTO'] - teste_feat['DATA_CADASTRO']
).dt.days

teste_feat['dias_prazo_cobranca'] = (
    teste_feat['DATA_VENCIMENTO'] - teste_feat['DATA_EMISSAO_DOCUMENTO']
).dt.days

teste_feat['mes_safra'] = teste_feat['DATA_VENCIMENTO'].dt.month
teste_feat['trimestre_safra'] = teste_feat['DATA_VENCIMENTO'].dt.quarter

print(" Features aplicadas na base de teste")


## 11.3 Tratamento de Valores Ausentes


**Objetivo:** Tratar valores ausentes na base de teste de forma consistente com a estratégia adotada no treinamento do modelo.

**Estratégia:**

        • Variáveis históricas (lags e agregações): Valores ausentes são preenchidos com 0, representando ausência de histórico (ex: clientes novos).

        • Variáveis numéricas: Imputação pela mediana, reduzindo o impacto de outliers e mantendo estabilidade estatística.

        • Variáveis categóricas: Valores ausentes são substituídos por 'DESCONHECIDO', preservando a informação de ausência.

**Importância:** A consistência no tratamento de missing entre treino e teste é essencial para evitar distorções nas previsões e garantir comportamento estável do modelo.

**Observação:** Essa abordagem mantém o significado original das variáveis e evita introdução de viés artificial nos dados.


In [ ]:

COLS_LAG = [
    'target_lag_1','target_lag_2','target_lag_3',
    'inad_ult_3','atraso_ult_3','atraso_ult_6',
    'taxa_hist_inad','atraso_hist_medio',
    'n_cobrancas_hist','valor_medio_cliente'
]

teste_feat[COLS_LAG] = teste_feat[COLS_LAG].fillna(0)

for col in teste_feat.select_dtypes(include='number').columns:
    if col not in COLS_LAG:
        teste_feat[col] = teste_feat[col].fillna(teste_feat[col].median())

for col in teste_feat.select_dtypes(include=['object','string']):
    teste_feat[col] = teste_feat[col].fillna('DESCONHECIDO')

print(" Dados tratados")

## 11.4 Predição do Modelo


**Objetivo:** Gerar as probabilidades de inadimplência para a base de teste, utilizando o modelo treinado.

**Estratégia:**

        • Seleção de features: São utilizadas exatamente as mesmas variáveis definidas na etapa de modelagem, garantindo alinhamento entre treino e inferência.

        • Tratamento de variáveis categóricas: As colunas categóricas são convertidas para o tipo 'category', mantendo compatibilidade com o modelo LightGBM.

        • Predição: O modelo gera probabilidades de inadimplência (classe 1), que serão utilizadas para priorização de risco.

**Importância:** A consistência na seleção de features e no tratamento dos dados garante que o modelo opere corretamente em produção, sem discrepâncias em relação ao treino.

**Observação:** O modelo retorna probabilidades, permitindo flexibilidade na definição de estratégias de decisão (ex: escolha de threshold).


In [ ]:

X_teste_final = teste_feat[FEATURES].copy()

for col in X_teste_final.select_dtypes(include=['object','string']):
    X_teste_final[col] = X_teste_final[col].astype('category')

prob_teste = model.predict_proba(X_teste_final)[:, 1]

print("Predições realizadas")

## 11.5 Geração do Arquivo Final


**Objetivo:** Gerar o arquivo final de submissão contendo as probabilidades de inadimplência para cada cliente e safra.

**Estratégia:**

        • Estrutura da submissão: O arquivo final contém as chaves identificadoras (ID_CLIENTE, SAFRA_REF) e a probabilidade prevista pelo modelo.

        • Garantia de unicidade: Remoção de possíveis duplicidades, assegurando uma única previsão por cliente/safra.

        • Exportação: O resultado é salvo em formato CSV, pronto para envio ou utilização em sistemas externos.

        • Validações finais: São exibidas estatísticas descritivas das probabilidades, permitindo verificar consistência e distribuição dos scores gerados.

**Importância:** Essa etapa consolida todo o pipeline desenvolvido, garantindo que o output esteja no formato correto, consistente e confiável para uso prático.

**Observação:** As probabilidades permitem flexibilidade na definição de estratégias, como priorização de clientes ou ajuste de thresholds conforme o contexto de negócio.


In [ ]:

submissao = teste_feat[['ID_CLIENTE','SAFRA_REF']].copy()
submissao['PROBABILIDADE_INADIMPLENCIA'] = prob_teste

submissao = submissao.drop_duplicates(['ID_CLIENTE','SAFRA_REF'])

submissao.to_csv('submissao_case.csv', index=False)

print(f"\nsubmissao_case.csv gerado!")
print(f"   {len(submissao):,} linhas")
print(submissao.describe())
print(f"\nProb média: {submissao['PROBABILIDADE_INADIMPLENCIA'].mean():.2%}")
print(f"% acima 0.5: {(submissao['PROBABILIDADE_INADIMPLENCIA'] > 0.5).mean():.2%}")

## 12. Regras de Negócio e Validação


**Objetivo:** Traduzir as probabilidades previstas em regras de ação, permitindo aplicação prática do modelo no processo de cobrança.

**Estratégia:**

    • Segmentação de risco: Os clientes são classificados em três níveis (baixo, médio e alto risco), com base na probabilidade de inadimplência.

    • Definição de thresholds: Os cortes foram definidos de forma heurística para segmentação inicial, podendo ser ajustados conforme estratégia de negócio.

    • Análise dos grupos: São avaliadas a distribuição e a probabilidade média por grupo, validando a coerência da segmentação.

**Importância:** Essa etapa conecta o modelo à operação, permitindo priorização de clientes, otimização de recursos e definição de estratégias de cobrança.

**Observação:** Os thresholds podem ser refinados com base em métricas de negócio, como custo de cobrança e taxa de recuperação.



In [ ]:

# Segmentação de Risco Baseada em Score
def classificar_risco(prob):
    if prob >= 0.6:
        return 'ALTO_RISCO'
    elif prob >= 0.3:
        return 'MEDIO_RISCO'
    else:
        return 'BAIXO_RISCO'


# Aplicar Regras de Negócio
submissao['REGRA_ACAO'] = submissao['PROBABILIDADE_INADIMPLENCIA'].apply(classificar_risco)


# Distribuição Geral
print("\nDistribuição das ações:")
print(submissao['REGRA_ACAO'].value_counts(normalize=True))


# Qualidade do Score por Grupo
print("\nProbabilidade média por grupo:")
print(submissao.groupby('REGRA_ACAO')['PROBABILIDADE_INADIMPLENCIA'].mean())


# Estatísticas Detalhadas
print("\nEstatísticas por grupo:")
print(submissao.groupby('REGRA_ACAO')['PROBABILIDADE_INADIMPLENCIA'].describe())


# Top Risco Operacional
print("\nTop 20 clientes com maior risco:")
display(submissao.sort_values('PROBABILIDADE_INADIMPLENCIA', ascending=False).head(20))


# Conclusão de Negócio
print("""
Interpretação:

O modelo segmenta os clientes em três níveis de risco (baixo, médio e alto), permitindo priorização eficiente das ações de cobrança.

Clientes de alto risco podem ser direcionados para ações mais intensivas, enquanto clientes de baixo risco podem seguir fluxo automatizado,
otimizando custo operacional e maximizando recuperação.

Os thresholds utilizados são iniciais e podem ser calibrados conforme estratégia e restrições do negócio.
""")

## Insights - Modelo de Risco de Inadimplência

### 1. Modelo com alta capacidade de separação de risco
O modelo apresentou forte performance preditiva:
- AUC-ROC ~0.91
- KS ~0.74

Indica excelente capacidade de ranquear clientes por risco.

---

### 2. Risco concentrado em uma pequena parcela da base
A distribuição das probabilidades mostra que poucos clientes concentram grande parte do risco.

Permite atuação direcionada e altamente eficiente.

---

### 3. Priorização operacional via ranking de risco
A análise de gains demonstra que uma fração reduzida da base captura grande parte dos inadimplentes.

Viabiliza estratégia de cobrança focada e redução de custos.

---

### 4. Segmentação acionável para negócio
O modelo foi convertido em regras operacionais:
- Baixo risco → fluxo automatizado
- Médio risco → cobrança leve
- Alto risco → atuação intensiva

Permite aplicação direta em processos de cobrança.

---

### 5. Histórico do cliente como principal driver de risco
As variáveis mais relevantes estão relacionadas ao comportamento passado:
- inadimplência anterior
- atrasos recentes
- padrão histórico

Confirma a importância do histórico na previsão de crédito.

---

### 6. Importância do comportamento recente
Variáveis de curto prazo (últimos períodos) têm maior impacto no risco.

Indica que deterioração recente é um forte sinal de inadimplência.